# PDF TO CHUNKS

In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

import pdfplumber
from pathlib import Path

pdf_path = Path("/Users/ruowen_vagabond/Desktop/Knowledge Graph - urban climate adaptation/data/text/50-climate-solutions-prc-cities.pdf")

pages_text = []

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            pages_text.append({
                "page": page_num + 1,
                "text": text
            })

print(len(pages_text))
print(pages_text[0]["text"][:500])

# clean page text data
import re
header_keywords = [
    "Best Practices from Cities",
    "Taking Action on Climate Change",
    "Asian Development Bank",
    "People’s Republic of China"
]
def remove_headers_footers(text):
    lines = text.split("\n")
    cleaned = []
    for line in lines:
        line_strip = line.strip()
        if not line_strip:
            continue
        if any(k.lower() in line_strip.lower() for k in header_keywords):
            continue
        if line_strip.isdigit():
            continue
        cleaned.append(line_strip)
    return "\n".join(cleaned)



def fix_hyphenation(text):
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    return text

def normalize_newlines(text):
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    text = re.sub(r'\n{2,}', '\n\n', text)
    return text

def normalize_spaces(text):
    return re.sub(r'[ \t]+', ' ', text)

def clean_page_text(text):
    text = fix_hyphenation(text)
    text = remove_headers_footers(text)
    text = normalize_newlines(text)
    text = normalize_spaces(text)
    return text.strip()

cleaned_pages = []

for page in pages_text:
    cleaned_text = clean_page_text(page["text"])
    if len(cleaned_text) > 200:  
        cleaned_pages.append({
            "page": page["page"],
            "text": cleaned_text
        })



# chunk split
chunk_size = 1000
chunk_overlap = 200

chunks = []

for page in cleaned_pages:
    text = page["text"]
    page_num = page["page"]
    
    start = 0
    chunk_id = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        
        chunks.append({
            "chunk_id": f"p{page_num}_c{chunk_id}",
            "page": page_num,
            "text": chunk_text.strip()
        })
        
        start += chunk_size - chunk_overlap
        chunk_id += 1





# ONTOLOGY EXTRACTION

In [ ]:
# ontology adjust
ONTOLOGY_v2 = {

    "entities": {

        "Document": {
            "description": "Official or authoritative document describing urban climate adaptation",
            "properties": [
                "title",
                "year",
                "publisher",
                "document_type",        
                "geographic_scope",     
                "source_url"
            ]
        },

        "Actor": {
            "description": "Organizations or groups involved in urban climate adaptation",
            "properties": [
                "name",
                "actor_type",           
                "actor_role",          
                "scale"                 
            ]
        },

        "City": {
            "description": "Urban area discussed or targeted in adaptation context",
            "properties": [
                "name",
                "country",
                "population",
                "GDP",
                "is_case_city",        
                "mentioned_role"       
            ]
        },

        "ClimateContext": {
            "description": "Climatic background of a city",
            "properties": [
                "climate_zone",
                "mean_temperature",
                "extreme_temperature",
                "precipitation_pattern"
            ]
        },

        "SocioEconomicContext": {
            "description": "Socio-economic background shaping adaptation capacity",
            "properties": [
                "education_level",
                "income_level",
                "development_status"
            ]
        },

        "Hazard": {
            "description": "Climate-related hazard affecting cities",
            "properties": [
                "hazard_type",          
                "temporal_pattern",     
                "climate_driver"      
            ]
        },

        "AdaptationStrategy": {
            "description": "High-level strategy addressing climate hazards",
            "properties": [
                "strategy_name",
                "sector",               
                "mechanism"           
            ]
        },

        "AdaptationAction": {
            "description": "Concrete implementation of an adaptation strategy",
            "properties": [
                "action_name",
                "implementation_status" 
            ]
        },

        "PlanningStage": {
            "description": "Stage of the planning and adaptation process",
            "properties": [
                "stage_name"            
            ]
        },

        "Target": {
            "description": "Urban system or population targeted by adaptation",
            "properties": [
                "target_type"          
            ]
        },

        "UrbanIndicator": {
            "description": "Urban environmental indicators shaping climate exposure",
            "properties": [
                "indicator_type",       
                "value",
                "data_source"
            ]
        },

        "PolicyNarrative": {
            "description": "Narrative framing used to justify adaptation strategies",
            "properties": [
                "narrative_type"       
            ]
        },

        "AdaptationOutcome": {
            "description": "Observed or expected outcomes of adaptation",
            "properties": [
                "outcome_type",         
                "direction",            
                "evidence_chunk"
            ]
        },

        "ImplementationCapacity": {
            "description": "Capacity of a city to implement adaptation actions",
            "properties": [
                "governance_structure",
                "funding_source",
                "financing_secured",   
                "monitoring_indicator"  
            ]
        }
    },

    # ========= Relationships =========
    "relations": {

        "PUBLISHED_BY": {
            "domain": "Document",
            "range": "Actor"
        },

        "DESCRIBES_CITY": {
            "domain": "Document",
            "range": "City"
        },

        "HAS_CLIMATE_CONTEXT": {
            "domain": "City",
            "range": "ClimateContext"
        },

        "HAS_SOCIOECONOMIC_CONTEXT": {
            "domain": "City",
            "range": "SocioEconomicContext"
        },

        "EXPOSED_TO": {
            "domain": "City",
            "range": "Hazard"
        },

        "ADDRESSES": {
            "domain": "AdaptationStrategy",
            "range": "Hazard"
        },

        "TARGETS": {
            "domain": "AdaptationStrategy",
            "range": "Target"
        },

        "IMPLEMENTED_BY": {
            "domain": "AdaptationAction",
            "range": "Actor"
        },

        "IMPLEMENTS": {
            "domain": "AdaptationAction",
            "range": "AdaptationStrategy"
        },

        "OCCURS_IN_STAGE": {
            "domain": "AdaptationAction",
            "range": "PlanningStage"
        },

        "INFLUENCES": {
            "domain": "AdaptationStrategy",
            "range": "UrbanIndicator"
        },

        "FRAMED_AS": {
            "domain": "AdaptationStrategy",
            "range": "PolicyNarrative"
        },

        "RESULTS_IN": {
            "domain": "AdaptationStrategy",
            "range": "AdaptationOutcome"
        },

        "AFFECTS": {
            "domain": "AdaptationOutcome",
            "range": "Target"
        },

        "HAS_IMPLEMENTATION_CAPACITY": {
            "domain": "City",
            "range": "ImplementationCapacity"
        }
    }
}



EXTRACTION_PROMPT_V2 = """
You are an expert research assistant specialized in urban climate adaptation,
urban planning, and climate policy analysis.

Your task is to extract structured knowledge from the text below,
strictly following the given ontology.

========================
ONTOLOGY
========================
{ontology}

========================
EXTRACTION RULES
========================

1. Extract ONLY information that is explicitly stated in the text.
   - Do NOT infer, assume, or generalize beyond the text.
   - If something is implied but not clearly stated, do NOT extract it.

2. Use the ontology as a schema, NOT as a closed vocabulary.
   - You may extract new entity names not listed in the ontology examples,
     as long as they clearly belong to one of the defined entity types.

3. For each extracted entity:
   - Assign a unique id.
   - Specify its entity type exactly as defined in the ontology.
   - Populate only the properties that are explicitly supported by the text.
   - Use null for properties that are not mentioned.

4. Extract relations ONLY when the text clearly expresses a relationship.
   - Each relation must have:
     - source (entity id)
     - relation_type (from the allowed relations)
     - target (entity id)

5. Allowed relation types include (but are not limited to):
   - INITIATES
   - IMPLEMENTS
   - PROMOTES
   - FRAMED_AS
   - ADDRESSES
   - TARGETS
   - OCCURS_IN
   - INVOLVES
   - RESULTS_IN

6. Policy narratives:
   - If the text describes a long-term vision, ideology, or framing
     (e.g., ecological civilization, green growth, win-win development),
     extract it as a PolicyNarrative entity.
   - If a strategy or action is explicitly described as aligned with,
     justified by, or framed within such a vision,
     extract a FRAMED_AS relation.

7. Evidence:
   - Always include the original source text chunk under
     "evidence.source_text".
   - Do NOT paraphrase or summarize the evidence text.

========================
OUTPUT FORMAT (JSON ONLY)
========================

{{
  "entities": [
    {{
      "id": "...",
      "type": "...",
      "properties": {{ }}
    }}
  ],
  "relations": [
    {{
      "source": "...",
      "relation_type": "...",
      "target": "..."
    }}
  ],
  "evidence": {{
    "source_text": "..."
  }}
}}

========================
TEXT TO ANALYZE
========================
{text}
"""



import json
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def extract_ontology_constrained_kg_v2(chunk_text: str, ontology: dict):
    prompt = EXTRACTION_PROMPT_V2.format(
        ontology=json.dumps(ontology, indent=2),
        text=chunk_text
    )
    response = llm.invoke(prompt)
    return json.loads(response.content)


test_chunk = chunks[15]["text"]

kg_json = extract_ontology_constrained_kg_v2(
    chunk_text=test_chunk,
    ontology=ONTOLOGY_v2
)

print(json.dumps(kg_json, indent=2, ensure_ascii=False))




In [ ]:
import json

output_path = "kg_output_chunk_15.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(kg_json, f, indent=2, ensure_ascii=False)

print(f"Saved KG extraction to {output_path}")

# KG CONSTRUCTION

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "urbanclimateadaptation"  
NEO4J_DATABASE = "climatekg" 

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

def merge_entity(tx, entity):
    
    label = entity["type"]
    entity_id = entity["id"]
    props = entity.get("properties", {})

    query = f"""
    MERGE (n:{label} {{id: $id}})
    SET n += $props
    """

    tx.run(
        query,
        id=entity_id,
        props=props
    )


def merge_relation(tx, relation, entity_lookup, evidence_text=None):
    source_id = relation["source"]
    target_id = relation["target"]
    rel_type = relation["relation_type"]

    source_entity = entity_lookup[source_id]
    target_entity = entity_lookup[target_id]

    source_label = source_entity["type"]
    target_label = target_entity["type"]

    query = f"""
    MATCH (a:{source_label} {{id: $source_id}})
    MATCH (b:{target_label} {{id: $target_id}})
    MERGE (a)-[r:{rel_type}]->(b)
    """

    params = {
        "source_id": source_id,
        "target_id": target_id
    }

    if evidence_text:
        query += """
        SET r.evidence = $evidence
        """
        params["evidence"] = evidence_text

    tx.run(query, **params)




In [ ]:
def instantiate_kg_from_json(kg_json):
    entities = kg_json["entities"]
    relations = kg_json.get("relations", [])
    evidence_text = kg_json.get("evidence", {}).get("source_text", "")

    entity_lookup = {e["id"]: e for e in entities}

    with driver.session(database=NEO4J_DATABASE) as session:
        session.execute_write(
            lambda tx: [merge_entity(tx, e) for e in entities]
        )

        session.execute_write(
            lambda tx: [
                merge_relation(tx, r, entity_lookup, evidence_text)
                for r in relations
            ]
        )

In [ ]:
instantiate_kg_from_json(kg_json)

In [ ]:
import time
import json

def extract_document_kg(chunks, ontology, sleep_sec=1.5):
    all_results = []

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}: {chunk['chunk_id']}")

        try:
            kg_json = extract_ontology_constrained_kg_v2(
                chunk_text=chunk["text"],
                ontology=ontology
            )

            kg_json["chunk_id"] = chunk["chunk_id"]
            kg_json["page"] = chunk["page"]

            all_results.append(kg_json)

            time.sleep(sleep_sec) 

        except Exception as e:
            print(f"Failed on chunk {chunk['chunk_id']}: {e}")

    return all_results


In [ ]:
doc_kg_json = extract_document_kg(
    chunks=chunks,
    ontology=ONTOLOGY_v2
)

In [ ]:
def merge_entity(tx, entity):
    label = entity["type"]
    props = entity["properties"]
    props["id"] = entity["id"]

    cypher = f"""
    MERGE (n:{label} {{id: $id}})
    SET n += $props
    """

    tx.run(cypher, id=entity["id"], props=props)

def merge_relation(tx, relation):
    cypher = """
    MATCH (a {id: $source}), (b {id: $target})
    MERGE (a)-[r:%s]->(b)
    """ % relation["relation_type"]

    tx.run(
        cypher,
        source=relation["source"],
        target=relation["target"]
    )


In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "neo4j://127.0.0.1:7687",
    auth=("neo4j", "urbanclimateadaptation")
)

def instantiate_document_kg(entities, relations):
    with driver.session(database="climatekg") as session:
        session.execute_write(
            lambda tx: [merge_entity(tx, e) for e in entities]
        )
        session.execute_write(
            lambda tx: [merge_relation(tx, r) for r in relations]
        )
